# Flat Region Plots

For each window in `valid_flats_df`, looks up the correct index in the full fold
training table (BED file) by matching `chrom / start / end`, then uses that index
to retrieve the right prediction and target from `targets_np / preds_np`.

**Saves for the first 10 windows in `valid_flats_df`:**
- Predicted contact map (with flat region box)
- Target contact map
- Predicted insulation profile (with flat region shaded)
- True insulation profile

All figures saved as SVG.

In [1]:
import sys, os, glob
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.abspath("/home1/smaruj/akita_semifreddo/"))
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

from akita.model import SeqNN
from utils.dataset_utils import HiCDataset
from utils.data_utils import from_upper_triu
from utils.insulation_utils import insulation_full

In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
SPECIES    = "mouse"   # "mouse" or "human"
FOLD       = 0
N_PLOT     = 10        # number of windows to save figures for

MATRIX_LEN   = 512
NUM_DIAGS    = 2
INSUL_WINDOW = 16
VMIN, VMAX   = -0.6, 0.6
CMAP         = "RdBu_r"

OUTPUT_DIR = "plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_PATH = {
    "mouse": "/home1/smaruj/pytorch_akita/models/finetuned/mouse/Hsieh2019_mESC/checkpoints/Akita_v2_mouse_Hsieh2019_mESC_model0_finetuned.pth",
    "human": "/home1/smaruj/pytorch_akita/models/finetuned/human/Krietenstein2019_HFF/checkpoints/Akita_v2_human_Krietenstein2019_HFF_model0_finetuned.pth",
}[SPECIES]

DATA_DIR = {
    "mouse": "/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/Akita_pytorch_training_data/mouse_training_data/Hsieh2019_mESC",
    "human": "/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/Akita_pytorch_training_data/human_training_data/Krietenstein2019_HFF",
}[SPECIES]

BED_FILE = {
    "mouse": "/project2/fudenber_735/tensorflow_models/akita/v2/data/mm10/sequences.bed",
    "human": "/project2/fudenber_735/tensorflow_models/akita/v2/data/hg38/sequences.bed",
}[SPECIES]

FLAT_REGIONS_TSV = (
    f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/"
    f"analysis/flat_regions/mouse_flat_regions_tsv/"
    f"fold{FOLD}_selected_genomic_windows_centered.tsv"
)

## 1. Load flat regions TSV and resolve indices into the fold training table

In [3]:
# Full BED table — one row per sequence across all folds
df_bed = pd.read_csv(BED_FILE, sep="\t", header=None, names=["chrom", "start", "end", "fold"])

# Keep only this fold and assign within-fold positional index (0, 1, 2, ...)
# This index matches the order sequences were loaded by HiCDataset
df_fold = df_bed[df_bed["fold"] == f"fold{FOLD}"].reset_index(drop=True)
df_fold["fold_idx"] = df_fold.index
print(f"fold{FOLD} has {len(df_fold)} sequences in the BED file")

# Load flat regions for this fold
valid_flats_df = pd.read_csv(FLAT_REGIONS_TSV, sep="\t")
print(f"Flat region windows: {len(valid_flats_df)}")
valid_flats_df.head()

fold0 has 725 sequences in the BED file
Flat region windows: 24


,chrom,start,end,fold,PearsonR,flat_start,flat_end,centered_start,centered_end,centered_flat_start,centered_flat_end
0,chr11,65921024,67231744,fold0,0.729526,60,199,65660928,66971648,187,325
1,chr3,38414336,39725056,fold0,0.674470,198,339,38438912,39749632,186,326
2,chr3,52123648,53434368,fold0,0.804358,105,246,51957760,53268480,186,326
3,chr3,99229696,100540416,fold0,0.688598,51,250,99012608,100323328,157,355
4,chr3,101851136,103161856,fold0,0.689428,205,315,101859328,103170048,201,311


In [4]:
# Match each flat region window back to its position in df_fold
# by joining on chrom / original start / original end
valid_flats_df = valid_flats_df.merge(
    df_fold[["chrom", "start", "end", "fold_idx"]],
    on=["chrom", "start", "end"],
    how="left",
)

missing = valid_flats_df["fold_idx"].isna().sum()
if missing > 0:
    print(f"Warning: {missing} windows could not be matched — check start/end columns")

valid_flats_df["fold_idx"] = valid_flats_df["fold_idx"].astype(int)
print(valid_flats_df[["chrom", "start", "end", "fold_idx", "flat_start", "flat_end"]].head(10))

   chrom      start        end  fold_idx  flat_start  flat_end
0  chr11   65921024   67231744        48          60       199
1   chr3   38414336   39725056       310         198       339
2   chr3   52123648   53434368       267         105       246
3   chr3   99229696  100540416       407          51       250
4   chr3  101851136  103161856       385         205       315
5   chr3  123035648  124346368        24         201       310
6   chr3  129916928  131227648       414          87       217
7   chr3  134412288  135723008       671         106       295
8   chr3  152762368  154073088       701          54       343
9   chr4   94595072   95905792        14         282       384


## 2. Load model and run inference on the full fold

In [5]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = SeqNN()
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device).eval()

data_files = sorted(glob.glob(os.path.join(DATA_DIR, f"fold{FOLD}_*.pt")))
print(f"Found {len(data_files)} data files")

dataset = HiCDataset(data_files)
loader  = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=4, pin_memory=True)

all_preds, all_targets = [], []
with torch.no_grad():
    for ohe_seq, hic_vec in loader:
        all_preds.append(model(ohe_seq.to(device)).cpu())
        all_targets.append(hic_vec.cpu())

all_preds   = torch.cat(all_preds,   dim=0)
all_targets = torch.cat(all_targets, dim=0)

targets_np = all_targets.numpy()
preds_np   = all_preds.numpy()
print(f"Loaded {len(preds_np)} maps for fold {FOLD}")

Device: cuda:0
Found 8 data files
  Loading: /project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/Akita_pytorch_training_data/mouse_training_data/Hsieh2019_mESC/fold0_0.pt
  Loading: /project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/Akita_pytorch_training_data/mouse_training_data/Hsieh2019_mESC/fold0_1.pt
  Loading: /project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/Akita_pytorch_training_data/mouse_training_data/Hsieh2019_mESC/fold0_2.pt
  Loading: /project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/Akita_pytorch_training_data/mouse_training_data/Hsieh2019_mESC/fold0_3.pt
  Loading: /project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/Akita_pytorch_training_data/mouse_training_data/Hsieh2019_mESC/fold0_4.pt
  Loading: /project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/Akita_pytorch_training_data/mouse_training_data/Hsieh2019_mESC/fold0_5.pt
  Loading: /project2/fudenber_735/sm

/home1/smaruj/miniconda3/envs/pytorch_hic/lib/python3.10/site-packages/torch/nn/modules/conv.py:306: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:84.)
  return F.conv1d(input, weight, bias, self.stride,


Loaded 725 maps for fold 0


## 3. Save figures for the first N windows

In [6]:
def save_contact_map(matrix, title, fname, flat_start=None, flat_end=None, out_dir=OUTPUT_DIR):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.matshow(matrix.astype(np.float32), cmap=CMAP, vmin=VMIN, vmax=VMAX)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title)
    if flat_start is not None and flat_end is not None:
        rect = patches.Rectangle(
            (flat_start, flat_start),
            flat_end - flat_start, flat_end - flat_start,
            linewidth=1.5, edgecolor="red", facecolor="none",
        )
        ax.add_patch(rect)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, fname), format="svg")
    plt.close()


def save_insulation_plot(insul, label, color, title, fname, flat_start=None, flat_end=None, out_dir=OUTPUT_DIR):
    x = np.arange(MATRIX_LEN)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(x, insul, color=color, label=label)
    ax.set_xlim(0, MATRIX_LEN)
    ax.set_ylim(VMIN, VMAX)
    ax.set_xlabel("Position (bins)")
    ax.set_ylabel("Insulation score")
    ax.set_title(title)
    ax.margins(x=0)
    if flat_start is not None and flat_end is not None:
        ax.axvspan(flat_start, flat_end, color="red", alpha=0.25, label="Flat region")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, fname), format="svg")
    plt.close()

In [7]:
plot_df = valid_flats_df.head(N_PLOT).reset_index(drop=True)

for i, row in plot_df.iterrows():
    fold_idx  = int(row["fold_idx"])
    flat_start = int(row["flat_start"]) if not pd.isna(row["flat_start"]) else None
    flat_end   = int(row["flat_end"])   if not pd.isna(row["flat_end"])   else None

    true_mat = from_upper_triu(targets_np[fold_idx], MATRIX_LEN, NUM_DIAGS)
    pred_mat = from_upper_triu(preds_np[fold_idx],   MATRIX_LEN, NUM_DIAGS)

    true_insul = insulation_full(true_mat, INSUL_WINDOW)
    pred_insul = insulation_full(pred_mat, INSUL_WINDOW)

    label = f"{row['chrom']}_{row['start']}"  # short label for filenames

    # Predicted map with flat region box
    save_contact_map(
        pred_mat,
        title=f"Predicted — {row['chrom']}:{row['start']} (fold_idx {fold_idx})",
        fname=f"pred_map_{label}.svg",
        flat_start=flat_start, flat_end=flat_end,
    )

    # Target map
    save_contact_map(
        true_mat,
        title=f"Target — {row['chrom']}:{row['start']} (fold_idx {fold_idx})",
        fname=f"target_map_{label}.svg",
    )

    # Predicted insulation
    save_insulation_plot(
        pred_insul, label="Pred", color="steelblue",
        title=f"Predicted insulation — {row['chrom']}:{row['start']}",
        fname=f"pred_insulation_{label}.svg",
        flat_start=flat_start, flat_end=flat_end,
    )

    # True insulation
    save_insulation_plot(
        true_insul, label="True", color="black",
        title=f"True insulation — {row['chrom']}:{row['start']}",
        fname=f"true_insulation_{label}.svg",
        flat_start=flat_start, flat_end=flat_end,
    )

    print(f"[{i+1}/{len(plot_df)}] Saved figures for {row['chrom']}:{row['start']} (fold_idx={fold_idx})")

print(f"\nAll figures saved to {OUTPUT_DIR}/")

[1/10] Saved figures for chr11:65921024 (fold_idx=48)
[2/10] Saved figures for chr3:38414336 (fold_idx=310)
[3/10] Saved figures for chr3:52123648 (fold_idx=267)
[4/10] Saved figures for chr3:99229696 (fold_idx=407)
[5/10] Saved figures for chr3:101851136 (fold_idx=385)
[6/10] Saved figures for chr3:123035648 (fold_idx=24)
[7/10] Saved figures for chr3:129916928 (fold_idx=414)
[8/10] Saved figures for chr3:134412288 (fold_idx=671)
[9/10] Saved figures for chr3:152762368 (fold_idx=701)
[10/10] Saved figures for chr4:94595072 (fold_idx=14)

All figures saved to plots/
